# 01 · Exploración longitudinal — Parkinson's Telemonitoring

| | |
|---|---|
| **Objetivo** | Entender la estructura longitudinal de los datos antes de modelar |
| **Pregunta guía** | ¿Cuánto de la variación en la gravedad es *entre* pacientes y cuánto *dentro* de cada paciente a lo largo del tiempo? |
| **Datos** | 42 pacientes con Parkinson temprano, grabaciones de voz en el hogar durante 6 meses ([UCI, CC BY 4.0](https://doi.org/10.24432/C5ZS3N)) |
| **Salida** | Hallazgos que definen cómo validar los modelos del notebook 02 |

**Índice**
0. Configuración
1. Resumen ejecutivo
2. Calidad de datos
3. Diseño longitudinal
4. Trayectorias de gravedad
5. Varianza entre y dentro de pacientes
6. Voz y gravedad
7. Redundancia entre medidas de voz
8. Conclusiones e implicancias para el modelado

## 0. Configuración

In [1]:
import sys
sys.path.append("..")  # permite importar desde src/

import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

from src.data import load_data
from src.analysis import (ID, TIEMPO, OBJETIVOS, columnas_voz, calidad_datos,
                          resumen_por_paciente, varianza_entre_pacientes,
                          centrar_por_paciente, correlaciones_con_objetivo,
                          pares_redundantes, metricas_clave, hallazgos_clave)
from src.viz import guardar, tarjetas_kpi, explorador_paciente, explorador_dispersion
pd.set_option("display.float_format", "{:.3f}".format)

OBJETIVO = "total_UPDRS"   # variable objetivo principal de este notebook

In [2]:
df = load_data()
df["paciente"] = df[ID].astype(str)   # versión texto, para colorear por paciente en los gráficos
VOZ = columnas_voz(df)
print(f"{df.shape[0]:,} registros · {df[ID].nunique()} pacientes · {len(VOZ)} medidas de voz")

5,875 registros · 42 pacientes · 16 medidas de voz


## 1. Resumen ejecutivo

Los indicadores y hallazgos se **calculan automáticamente** a partir de los datos (ver `src/analysis.py`),
así el resumen siempre coincide con el análisis de abajo.

In [3]:
m = metricas_clave(df, OBJETIVO)

kpis = tarjetas_kpi([
    ("Pacientes", m["n_pacientes"], ",.0f"),
    ("Registros", m["n_registros"], ",.0f"),
    ("Registros por paciente (mediana)", m["registros_mediana"], ",.0f"),
    ("Varianza entre pacientes", m["icc_objetivo"], ".0%"),
])
guardar(kpis, "01_kpis")
kpis.show()

display(Markdown("### Hallazgos clave\n\n" +
                 "\n".join(f"{i}. {h}" for i, h in enumerate(hallazgos_clave(m, OBJETIVO), 1))))

### Hallazgos clave

1. **Diseño longitudinal denso:** 42 pacientes y 5,875 registros (mediana de 141 por paciente, seguidos ~171 días). Hay suficientes mediciones por paciente para modelos de secuencias.
2. **La gravedad es casi una 'firma' del paciente:** el 93% de la varianza de `total_UPDRS` se explica por quién es el paciente. Dividir train/test al azar inflaría las métricas: la partición debe ser **por paciente**.
3. **Progresión lenta en 6 meses:** el cambio mediano de `total_UPDRS` fue de +4.2 puntos y el 69% de los pacientes empeoró. Predecir el *cambio* es más difícil que predecir el *nivel*.
4. **Asociación débil entre voz y gravedad:** la variable más correlacionada es `HNR` (Spearman global -0.16), y ninguna medida de voz por sí sola muestra una asociación fuerte, ni global ni dentro del paciente: habrá que evaluar combinaciones de variables en los modelos. Además hay 22 pares de medidas de voz casi redundantes (|r| ≥ 0.9).

## 2. Calidad de datos

**Por qué:** antes de analizar, confirmar tipos, faltantes y rangos. En datos clínicos no se elimina
nada sin justificarlo: se **documenta**.

In [4]:
calidad_datos(df.drop(columns="paciente"))

,tipo,faltantes,unicos,min,max
subject_id,int64,0,42,1.000,42.000
age,int64,0,23,36.000,85.000
sex,int64,0,2,0.000,1.000
test_time,float64,0,2442,-4.263,215.490
motor_UPDRS,float64,0,1080,5.038,39.511
total_UPDRS,float64,0,1129,7.000,54.992
Jitter(%),float64,0,1305,0.001,0.100
Jitter(Abs),float64,0,4105,0.000,0.000
Jitter:RAP,float64,0,853,0.000,0.058
Jitter:PPQ5,float64,0,840,0.000,0.070


In [5]:
duplicados = df.duplicated().sum()
negativos = (df[TIEMPO] < 0).sum()
print(f"Filas duplicadas: {duplicados}")
print(f"Registros con {TIEMPO} < 0: {negativos} "
      f"(mediciones previas a la fecha formal de inicio; se conservan y se documentan)")

Filas duplicadas: 0
Registros con test_time < 0: 12 (mediciones previas a la fecha formal de inicio; se conservan y se documentan)


## 3. Diseño longitudinal

**Por qué:** la cantidad de mediciones por paciente y la duración del seguimiento definen qué técnicas son posibles
(con pocas visitas no tiene sentido un LSTM; con muchas, sí).

Pasamos de **formato largo** (una fila por medición) a **una fila por paciente**:

In [6]:
pacientes = resumen_por_paciente(df, OBJETIVO)
pacientes.head()

,n_registros,dia_min,dia_max,edad,sexo,updrs_inicial,updrs_final,seguimiento_dias,cambio_updrs
subject_id,,,,,,,,,
1,149,5.643,174.660,72,Hombre,34.398,47.970,169.017,13.572
2,145,3.866,178.800,58,Hombre,14.039,21.650,174.934,7.611
3,144,7.288,168.270,57,Hombre,25.729,39.947,160.982,14.218
4,137,9.687,170.760,74,Hombre,15.263,27.770,161.073,12.507
5,156,7.360,168.390,75,Hombre,39.240,44.761,161.030,5.521


In [7]:
pacientes[["n_registros", "seguimiento_dias", "edad", "updrs_inicial", "cambio_updrs"]].describe().T

,count,mean,std,min,25%,50%,75%,max
n_registros,42.000,139.881,15.543,101.000,130.000,141.000,149.750,168.000
seguimiento_dias,42.000,172.514,10.713,153.008,167.285,170.517,175.044,203.995
edad,42.000,64.405,9.240,36.000,58.000,65.000,72.000,85.000
updrs_inicial,42.000,26.598,10.615,7.955,18.615,25.793,34.120,54.073
cambio_updrs,42.000,3.247,7.242,-15.813,-0.860,4.163,7.850,15.656


In [8]:
fig = px.histogram(pacientes, x="n_registros", nbins=20,
                   title="Cantidad de registros por paciente",
                   labels={"n_registros": "Registros por paciente"})
fig.update_layout(yaxis_title="Pacientes", bargap=0.05)
guardar(fig, "01_registros_por_paciente")
fig

In [9]:
fig = px.scatter(pacientes.reset_index(), x="updrs_inicial", y="cambio_updrs", color="sexo",
                 hover_data=[ID, "edad", "n_registros"],
                 title=f"¿Los pacientes más graves al inicio empeoran más? ({OBJETIVO})",
                 labels={"updrs_inicial": f"{OBJETIVO} inicial", "cambio_updrs": "Cambio en el período"})
fig.add_hline(y=0, line_dash="dash", line_color="gray")
guardar(fig, "01_inicial_vs_cambio")
fig

## 4. Trayectorias de gravedad

**Por qué:** el *spaghetti plot* (una línea por paciente) es el gráfico central en datos longitudinales.
Muestra si los pacientes **arrancan en niveles distintos** y si **evolucionan distinto**.

> ⚠️ Según el estudio original, la UPDRS fue evaluada por un médico al inicio, a los 3 y a los 6 meses;
> los valores intermedios fueron interpolados. Por eso las trayectorias se ven casi rectas.

In [10]:
fig = px.line(df, x=TIEMPO, y=OBJETIVO, color="paciente", line_group="paciente",
              title=f"Trayectoria de {OBJETIVO} por paciente",
              labels={TIEMPO: "Días desde el inicio"})
fig.update_traces(opacity=0.6, line_width=1.2)
fig.update_layout(showlegend=False)
guardar(fig, "01_trayectorias")
fig

**Explorador interactivo:** usá el menú desplegable (arriba a la derecha) para elegir un paciente y ver su evolución motora y total.

In [11]:
fig = explorador_paciente(df, ID, TIEMPO, OBJETIVOS)
guardar(fig, "01_explorador_paciente")
fig

## 5. Varianza entre y dentro de pacientes ⭐

**Por qué:** es el hallazgo que define cómo validar los modelos. Si casi toda la varianza es *entre* pacientes,
un modelo con partición aleatoria aprende a **reconocer al paciente** en lugar de aprender predictores reales,
y las métricas se inflan (el mismo problema que la partición temporal en el proyecto de Vaca Muerta).

**Cómo se calcula:** proporción = varianza de los promedios por paciente / varianza total
(una aproximación al coeficiente de correlación intraclase, ICC).

In [20]:
orden = df.groupby("paciente")[OBJETIVO].median().sort_values().index
fig = px.box(df, x="paciente", y=OBJETIVO, category_orders={"paciente": list(orden)},
             title=f"Distribución de {OBJETIVO} por paciente (ordenados por mediana)")
fig.update_layout(xaxis_title="Paciente")
guardar(fig, "01_box_por_paciente")
fig

In [13]:
varianzas = varianza_entre_pacientes(df, OBJETIVOS + VOZ)
varianzas

,entre_pacientes,dentro_paciente
variable,,
total_UPDRS,0.934,0.066
motor_UPDRS,0.918,0.082
DFA,0.769,0.231
HNR,0.606,0.394
Shimmer:APQ5,0.566,0.434
NHR,0.562,0.438
Shimmer(dB),0.553,0.447
Shimmer,0.552,0.448
Shimmer:APQ11,0.527,0.473


In [14]:
fig = px.bar(varianzas.reset_index().melt(id_vars="variable", var_name="componente", value_name="proporcion"),
             y="variable", x="proporcion", color="componente", orientation="h",
             title="Proporción de la varianza entre pacientes vs. dentro de cada paciente",
             labels={"proporcion": "Proporción de la varianza", "variable": ""},
             color_discrete_map={"entre_pacientes": "#4C78A8", "dentro_paciente": "#F58518"})
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=600, xaxis_tickformat=".0%")
guardar(fig, "01_varianza_entre_dentro")
fig

## 6. Voz y gravedad

Calculamos la correlación de Spearman (no asume relación lineal y es robusta a valores extremos) en **dos niveles**:

- **Global:** mezcla las diferencias entre pacientes con los cambios en el tiempo.
- **Dentro del paciente:** a cada valor se le resta el promedio de su paciente. Responde la pregunta longitudinal:
  *"cuando la voz de ESTE paciente cambia, ¿cambia su UPDRS?"*

Si la correlación global es alta pero la de dentro del paciente es baja, la voz sirve para **distinguir pacientes**
pero no para **seguir su evolución**.

In [15]:
correlaciones = correlaciones_con_objetivo(df, OBJETIVO)
correlaciones

,global,dentro_paciente
HNR,-0.162,0.029
Shimmer:APQ11,0.161,-0.030
PPE,0.155,-0.009
RPDE,0.150,0.001
NHR,0.144,-0.013
DFA,-0.142,-0.015
Shimmer(dB),0.140,-0.021
Shimmer,0.138,-0.021
Jitter(%),0.129,-0.009
Shimmer:APQ5,0.125,-0.023


In [16]:
fig = px.bar(correlaciones.reset_index(names="variable").melt(id_vars="variable", var_name="nivel", value_name="spearman"),
             y="variable", x="spearman", color="nivel", barmode="group", orientation="h",
             title=f"Correlación de Spearman con {OBJETIVO}: global vs. dentro del paciente",
             labels={"spearman": "Correlación de Spearman", "variable": ""})
fig.add_vline(x=0, line_color="black", line_width=1)
fig.update_layout(height=650, yaxis={"autorange": "reversed"})
guardar(fig, "01_correlaciones_voz")
fig

**Explorador interactivo:** elegí una medida de voz en el menú. A la izquierda, la relación global; a la derecha, la misma relación dentro de cada paciente (valores centrados en su propio promedio).

In [17]:
centrados = centrar_por_paciente(df, VOZ + [OBJETIVO]).assign(**{ID: df[ID]})
fig = explorador_dispersion(df, centrados, list(correlaciones.index), OBJETIVO, correlaciones, ID)
guardar(fig, "01_explorador_voz")
fig

## 7. Redundancia entre medidas de voz

**Por qué:** las variantes de jitter (tono) y de shimmer (volumen) miden casi lo mismo. Esa multicolinealidad
complica la interpretación en modelos lineales y reparte la importancia entre variables "gemelas" en los modelos de árboles.

In [18]:
corr_voz = df[VOZ].corr(method="spearman")
fig = px.imshow(corr_voz, color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="auto",
                title="Correlación de Spearman entre medidas de voz")
fig.update_layout(height=650)
guardar(fig, "01_heatmap_voz")
fig

In [19]:
redundantes = pares_redundantes(df, umbral=0.9)
print(f"{len(redundantes)} pares con |r| ≥ 0.9")
redundantes

22 pares con |r| ≥ 0.9


,var_1,var_2,r
0,Shimmer:APQ3,Shimmer:DDA,1.000
1,Jitter:RAP,Jitter:DDP,1.000
2,Shimmer,Shimmer:APQ5,0.990
3,Shimmer,Shimmer(dB),0.986
4,Shimmer,Shimmer:DDA,0.983
5,Shimmer,Shimmer:APQ3,0.983
6,Shimmer:APQ5,Shimmer:DDA,0.981
7,Shimmer:APQ3,Shimmer:APQ5,0.981
8,Shimmer(dB),Shimmer:APQ5,0.972
9,Shimmer(dB),Shimmer:APQ3,0.963


## 8. Conclusiones e implicancias para el modelado

**Lo que encontré**

1. **Diseño longitudinal denso.** 42 pacientes (28 hombres, 14 mujeres; edad mediana 65 años) con 5.875 registros: entre 101 y 168 por paciente (mediana 141), seguidos durante unos 171 días. No hay faltantes ni duplicados. Hay 12 registros con `test_time` negativo, de 2 pacientes; los conservo y los documento. La cantidad de mediciones por paciente alcanza para probar modelos de secuencias.

2. **La gravedad es casi una "firma" del paciente.** El 93% de la varianza de `total_UPDRS` (92% en `motor_UPDRS`) se explica por quién es el paciente; solo el 7% es variación dentro de cada uno a lo largo del tiempo. En cambio, las medidas de voz varían mucho dentro del mismo paciente: entre el 23% y el 73% de su varianza es intra-paciente, es decir, son mediciones ruidosas de una grabación a otra. **Implicancia:** una partición aleatoria de train/test permitiría que el modelo "reconozca al paciente" y las métricas quedarían infladas.

3. **Progresión heterogénea en ~6 meses.** El cambio mediano de `total_UPDRS` fue de +4,2 puntos, pero con mucha dispersión: de −15,8 a +15,7. 29 pacientes empeoraron (69%) y 13 mejoraron. La gravedad inicial no predice cuánto empeora cada paciente (Spearman −0,12; p = 0,46).

4. **La voz distingue entre pacientes, pero no sigue la evolución de cada uno.** A nivel global, las asociaciones con `total_UPDRS` son débiles: la más fuerte es `HNR` (−0,16), seguida de `Shimmer:APQ11` (+0,16) y `PPE` (+0,16). Dentro de cada paciente, todas las correlaciones caen prácticamente a cero (|r| ≤ 0,03). Como referencia, la edad sola correlaciona más con la gravedad (+0,32) que cualquier medida de voz. Además hay 22 pares de medidas casi redundantes (|r| ≥ 0,9), dentro de las familias Jitter y Shimmer.

**Limitaciones**
- La UPDRS fue evaluada clínicamente solo al inicio, a los 3 y a los 6 meses, y los valores intermedios están interpolados. Esto reduce artificialmente la variación real dentro de cada paciente y probablemente explica en parte las correlaciones intra-paciente cercanas a cero.
- 42 pacientes, todos en etapa temprana: muestra chica para generalizar.
- No hay variables autonómicas (termorregulación). Para esa pregunta, el siguiente paso sería PPMI con la escala SCOPA-AUT.

**Decisiones para el notebook 02**
- Validar con `GroupKFold` por paciente y compararlo contra una partición aleatoria, para cuantificar la fuga de información.
- Incluir edad y sexo como predictores base. Cualquier modelo de voz tiene que superar a un modelo que solo usa edad.
- Reducir la redundancia (quedarme con una o dos medidas por familia) o usar modelos robustos a la multicolinealidad.
- Promediar las grabaciones por semana para reducir el ruido de las medidas de voz.
- Separar dos tareas: predecir el **nivel** de gravedad de un paciente nuevo y predecir el **cambio** a lo largo del tiempo.